In [1]:
import pandas as pd
import numpy as np
from urllib.request import urlopen
import certifi
import json
from fredapi import Fred
import os
import ssl

# Custom packages
import derive_data as dd

import warnings
warnings.filterwarnings("ignore")

# Environment variables
import dotenv
dotenv.load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")
FMP_API_KEY = os.getenv("FMP_API_KEY")

In [2]:
fmp_idx = pd.read_csv('data/inputs/fmp_index_list.csv')
fmp_comm = pd.read_csv('data/inputs/fmp_commodity_list.csv')
us_equity_symbol_names = {
    # BOND ETFS
    'BIL': 'SPDR Bloomberg 1-3 Month T-Bill ETF',
    'SHY': 'iShares 1-3 Year Treasury Bond ETF',
    'IEF': 'iShares 7-10 Year Treasury Bond ETF',
    # MAJOR INDICES
    '^GSPC': 'S&P 500',
    '^IXIC': 'Nasdaq Composite',
    '^NDX': 'Nasdaq 100',
    '^RUT': 'Russell 2000',
    '^DJI': 'Dow Jones Industrial Average',
    '^RUI': 'Russell 1000',
    '^RUA': 'Russell 3000',
    
    # MAIN BROAD MARKET ETFS
    'SPY': 'SPDR S&P 500 ETF',
    'VOO': 'Vanguard S&P 500 ETF',
    'RSP': 'Invesco S&P 500 Equal Weight ETF',
    'IVV': 'iShares Core S&P 500 ETF',
    'QQQ': 'Invesco QQQ Trust',
    'QQQM': 'Invesco Nasdaq 100 ETF',
    'ONEQ': 'Fidelity Nasdaq Composite Index ETF',
    'IWM': 'iShares Russell 2000 ETF',
    'IWB': 'iShares Russell 1000 ETF',
    'IWV': 'iShares Russell 3000 ETF',
    'DIA': 'SPDR Dow Jones Industrial Average ETF',
    'VTI': 'Vanguard Total Stock Market ETF',
    
    # S&P 500 SECTOR ETFS (SELECT SECTOR SPDRS)
    'XLE': 'Energy Select Sector SPDR',
    'XLF': 'Financial Select Sector SPDR',
    'XLU': 'Utilities Select Sector SPDR',
    'XLI': 'Industrial Select Sector SPDR',
    'XLV': 'Health Care Select Sector SPDR',
    'XLK': 'Technology Select Sector SPDR',
    'XLB': 'Materials Select Sector SPDR',
    'XLY': 'Consumer Discretionary Select Sector SPDR',
    'XLP': 'Consumer Staples Select Sector SPDR',
    'XLRE': 'Real Estate Select Sector SPDR',
    'XLC': 'Communication Services Select Sector SPDR',
    
    # GROWTH ETFs
    'IVW': 'iShares S&P 500 Growth ETF',
    'VONG': 'Vanguard Russell 1000 Growth ETF',
    'IWF': 'iShares Russell 1000 Growth ETF',
    'IWO': 'iShares Russell 2000 Growth ETF',
    'VUG': 'Vanguard Growth ETF',
    'SPYG': 'SPDR Portfolio S&P 500 Growth ETF',
    
    # VALUE ETFs
    'IVE': 'iShares S&P 500 Value ETF',
    'VONV': 'Vanguard Russell 1000 Value ETF',
    'IWD': 'iShares Russell 1000 Value ETF',
    'IWN': 'iShares Russell 2000 Value ETF',
    'VTV': 'Vanguard Value ETF',
    'SPYV': 'SPDR Portfolio S&P 500 Value ETF',
    
    # SIZE ETFs
    'IWR': 'iShares Russell Mid-Cap ETF',
    'IWC': 'iShares Micro-Cap ETF',
    'IJH': 'iShares Core S&P Mid-Cap ETF',
    'IJR': 'iShares Core S&P Small-Cap ETF',
    'MDY': 'SPDR S&P MidCap 400 ETF',
    'SLY': 'SPDR S&P 600 Small Cap ETF',
    'VO': 'Vanguard Mid-Cap ETF',
    'VB': 'Vanguard Small-Cap ETF',
    'SCHA': 'Schwab U.S. Small-Cap ETF',
    'SCHM': 'Schwab U.S. Mid-Cap ETF',
    'VTWO': 'Vanguard Russell 2000 ETF',
    'VTHR': 'Vanguard Russell 3000 ETF',
    'THRK': 'iShares Russell 3000 ETF',
    'SPSM': 'SPDR Portfolio S&P 600 Small Cap ETF',
    'SMLF': 'iShares Small-Cap US Equity Factor ETF',
    
    # NASDAQ SPECIFIC
    'QTEC': 'First Trust Nasdaq-100 Technology Sector Index Fund',
    'QQEW': 'First Trust Nasdaq-100 Equal Weighted Index Fund',
    'QQQG': 'Pacer Nasdaq 100 Top 50 Cash Cows Dividend Growth ETF',
    'QQQV': 'Pacer Nasdaq 100 Top 50 Value ETF',
    
    # DIVIDEND/QUALITY
    'SCHD': 'Schwab U.S. Dividend Equity ETF',
    'VYM': 'Vanguard High Dividend Yield ETF',
    'DVY': 'iShares Select Dividend ETF',
    'QUAL': 'iShares MSCI USA Quality Factor ETF',
    'USMV': 'iShares MSCI USA Min Vol Factor ETF',
    
    # EQUAL WEIGHT
    'EWSC': 'Invesco S&P SmallCap 600 Equal Weight ETF',
    'EWMC': 'Invesco S&P MidCap 400 Equal Weight ETF',
}
int_equity_symbol_names = {
    'VXUS': 'Vanguard Total International Stock ETF',
    'VEA': 'Vanguard FTSE Developed Markets ETF',
    'VWO': 'Vanguard FTSE Emerging Markets ETF',
    'VGK': 'Vanguard FTSE Europe ETF',
    'VPL': 'Vanguard FTSE Pacific ETF',
    'FXI': 'iShares China Large-Cap ETF',
    'EWJ': 'iShares MSCI Japan ETF',
    'INDA': 'iShares MSCI India ETF',
}
macro_codes_dict = {
    # RATE BENCHMARKS
    'FEDFUNDS': 'Federal Funds Effective Rate',
    'SOFR': 'Secured Overnight Financing Rate',
    # TREASURY RATES
    'DGS1MO': '1-Month Treasury Rate',
    'DGS3MO': '3-Month Treasury Rate', 
    'DGS6MO': '6-Month Treasury Rate',
    'DGS1': '1-Year Treasury Rate',
    'DGS2': '2-Year Treasury Rate',
    'DGS3': '3-Year Treasury Rate',
    'DGS5': '5-Year Treasury Rate',
    'DGS7': '7-Year Treasury Rate',
    'DGS10': '10-Year Treasury Rate',
    'DGS20': '20-Year Treasury Rate',
    'DGS30': '30-Year Treasury Rate',
    # OTHER RATES
    'AAA': 'Moody\'s Seasoned AAA Corporate Bond Yield',
    'BAA': 'Moody\'s Seasoned BAA Corporate Bond Yield',
    'MORTGAGE30US': '30-Year Fixed Rate Mortgage Average',
    'DPRIME': 'Bank Prime Loan Rate',
    'T5YIE': '5-Year Breakeven Inflation Rate',
    'T10YIE': '10-Year Breakeven Inflation Rate',
    'T30YIE': '30-Year Breakeven Inflation Rate',
    # ECONOMIC INDICATORS
    'GDP': 'Gross Domestic Product',
    'GDPC1': 'Real Gross Domestic Product',
    'A939RX0Q048SBEA': 'Real GDP Per Capita',
    'PCE': 'Personal Consumption Expenditures',
    'PCEPI': 'Personal Consumption Expenditures Price Index',
    'PCEC96': 'Real Personal Consumption Expenditures',
    'CPIAUCSL': 'Consumer Price Index',
    'CPILFESL': 'Core CPI (Less Food and Energy)',
    'UNRATE': 'Unemployment Rate',
    'CIVPART': 'Labor Force Participation Rate',
    'INDPRO': 'Industrial Production Index',
    'PAYEMS': 'Total Nonfarm Payrolls',
    'HOUST': 'Housing Starts',
    'PERMIT': 'Building Permits',
    'MTSDS133FMS': 'Monthly US Government Surplus/Deficit',
    'GFDEGDQ188S': 'Federal Government Debt to GDP Ratio',
    'PMSAVE': 'Personal Savings',
    'PSAVERT': 'Personal Saving Rate',
    'GPDI': 'Gross Private Domestic Investment',
    'GPDIC1': 'Real Gross Private Domestic Investment',
    'BOGZ1FU263092001Q': 'Foreign Direct Investment in the United States',
    'QBPBSTAS': 'Balance Sheet - Total Assets',
    'FDHBFRBN': 'Federal Debt Held by Federal Reserve Banks',
    'FYGFDPUN': 'Federal Debt Held by the Public',
    'FDHBFIN': 'Federal Debt Held by Foreign Investors',
    'COMPOUT': 'Commercial Paper Outstanding',
    'ABCOMP': 'Asset-Backed Commercial Paper Outstanding',
    # MONEY SUPPLY
    'M1SL': 'M1 Money Stock',
    'M2SL': 'M2 Money Stock',
    'BASE': 'St. Louis Adjusted Monetary Base',
    # MARKET INDICATORS
    'VIXCLS': 'CBOE Volatility Index (VIX)',
    'UMCSENT': 'University of Michigan Consumer Sentiment',
    'USSLIND': 'Leading Index for the United States',
    'VISASMIHSA': 'Visa U.S. Consumer Spending Momentum Index: Headline',
    'VISASMIDSA': 'Visa U.S. Consumer Spending Momentum Index: Discretionary',
}
macro_units = {
    # INTEREST RATES AND FINANCIAL RATES
    'FEDFUNDS': 'Percent, Seasonally Adjusted',
    'SOFR': 'Percent, Not Seasonally Adjusted',
    'DGS1MO': 'Percent, Not Seasonally Adjusted',
    'DGS3MO': 'Percent, Not Seasonally Adjusted',
    'DGS6MO': 'Percent, Not Seasonally Adjusted',
    'DGS1': 'Percent, Not Seasonally Adjusted',
    'DGS2': 'Percent, Not Seasonally Adjusted',
    'DGS3': 'Percent, Not Seasonally Adjusted',
    'DGS5': 'Percent, Not Seasonally Adjusted',
    'DGS7': 'Percent, Not Seasonally Adjusted',
    'DGS10': 'Percent, Not Seasonally Adjusted',
    'DGS20': 'Percent, Not Seasonally Adjusted',
    'DGS30': 'Percent, Not Seasonally Adjusted',
    'AAA': 'Percent, Not Seasonally Adjusted',
    'BAA': 'Percent, Not Seasonally Adjusted',
    'MORTGAGE30US': 'Percent, Not Seasonally Adjusted',
    'DPRIME': 'Percent, Not Seasonally Adjusted',
    'T5YIE': 'Percent, Not Seasonally Adjusted',
    'T10YIE': 'Percent, Not Seasonally Adjusted',
    'T30YIE': 'Percent, Not Seasonally Adjusted',
    # ECONOMIC INDICATORS
    'GDP': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'GDPC1': 'Billions of Chained 2017 Dollars, Seasonally Adjusted Annual Rate',
    'A939RX0Q048SBEA': 'Chained 2017 Dollars, Seasonally Adjusted',
    'PCE': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'PCEPI': 'Index 2017=100, Seasonally Adjusted',
    'PCEC96': 'Billions of Chained 2017 Dollars, Seasonally Adjusted',
    'CPIAUCSL': 'Index 1982-1984=100, Seasonally Adjusted',
    'CPILFESL': 'Index 1982-1984=100, Seasonally Adjusted',
    'UNRATE': 'Percent, Seasonally Adjusted',
    'CIVPART': 'Percent, Seasonally Adjusted',
    'INDPRO': 'Index 2017=100, Seasonally Adjusted',
    'PAYEMS': 'Thousands of Persons, Seasonally Adjusted',
    'HOUST': 'Thousands of Units, Seasonally Adjusted Annual Rate',
    'PERMIT': 'Thousands of Units, Seasonally Adjusted Annual Rate',
    'MTSDS133FMS': 'Millions of Dollars, Not Seasonally Adjusted',
    'GFDEGDQ188S': 'Percent of GDP, Not Seasonally Adjusted',
    'PMSAVE': 'Billions of Dollars, Not Seasonally Adjusted',
    'PSAVERT': 'Percent, Seasonally Adjusted',
    'GPDI': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'GPDIC1': 'Billions of Chained 2017 Dollars, Seasonally Adjusted Annual Rate',
    'BOGZ1FU263092001Q': 'Millions of Dollars, Not Seasonally Adjusted',
    'QBPBSTAS': 'Millions of Dollars, Not Seasonally Adjusted',
    'FDHBFRBN': 'Billions of Dollars, Not Seasonally Adjusted',
    'FYGFDPUN': 'Millions of Dollars, Not Seasonally Adjusted',
    'FDHBFIN': 'Billions of Dollars, Not Seasonally Adjusted',
    'COMPOUT': 'Billions of Dollars, Not Seasonally Adjusted',
    'ABCOMP': 'Billions of Dollars, Not Seasonally Adjusted',
    # MONEY SUPPLY
    'M1SL': 'Billions of Dollars, Seasonally Adjusted',
    'M2SL': 'Billions of Dollars, Seasonally Adjusted',
    'BASE': 'Millions of Dollars, Not Seasonally Adjusted',
    # MARKET INDICATORS
    'VIXCLS': 'Index, Not Seasonally Adjusted',
    'UMCSENT': 'Index 1966:Q1=100, Not Seasonally Adjusted',
    'USSLIND': 'Percent, Seasonally Adjusted',
    'VISASMIHSA': 'Index, Seasonally Adjusted',
    'VISASMIDSA': 'Index, Seasonally Adjusted',
}


# US Equities

In [5]:
us_equity_data = pd.read_csv('data/processed/us_equity_all_data.csv', index_col=0, header=[0, 1], parse_dates=True)
us_equity_data.index = pd.to_datetime(us_equity_data.index)
us_equity_data.rename(columns=us_equity_symbol_names, level=0, inplace=True)

us_equity_always_disclude = ['Vanguard S&P 500 ETF', 'Real Estate Select Sector SPDR', 'Communication Services Select Sector SPDR']
us_equity_disclude = [name for name in us_equity_data.columns.get_level_values(0) if 'Russell 1000' in name or 'Russell 3000' in name]\
                        + ['S&P 500', 'Nasdaq Composite', 'Dow Jones Industrial Average', 'Nasdaq 100', 'Russell 2000']
us_treasuries = ['SPDR Bloomberg 1-3 Month T-Bill ETF', 'iShares 1-3 Year Treasury Bond ETF', 'iShares 7-10 Year Treasury Bond ETF']

us_equity_include = list(set(us_equity_data.columns.get_level_values(0).tolist()) - set(us_equity_always_disclude)\
                          - set(us_equity_disclude) - set(int_equity_symbol_names.keys()) - set(us_treasuries))
col_mask = us_equity_data.columns.map(lambda x: x[0] in us_equity_include)

us_equity_data = us_equity_data.loc[:, col_mask]
us_equity_data.tail()

SPDR S&P 500 ETF                                               \
                       open    high     low   close      volume     vwap   
date                                                                       
2025-10-20           667.32  672.21  667.27  671.30  60493400.0  669.740   
2025-10-21           671.44  672.99  669.98  671.29  56249034.0  671.485   
2025-10-22           672.00  672.00  663.30  667.80  80564006.0  667.650   
2025-10-23           668.12  672.71  667.80  671.76  65604500.0  670.255   
2025-10-24           676.46  678.47  675.65  677.25  74356527.0  677.060   

           Invesco S&P 500 Equal Weight ETF                          ...  \
                                       open    high     low   close  ...   
date                                                                 ...   
2025-10-20                           189.02  190.16  189.00  189.89  ...   
2025-10-21                           189.86  191.33  189.63  190.76  ...   
2025-10-22                           190.84  191.12  189.16  189.84  ...   
2025-10-23                           190.12  191.09  189.58  190.74  ...   
2025-10-24                           192.11  192.22  191.18  191.26  ...   

           iShares Russell Mid-Cap ETF                            \
                                   low  close     volume    vwap   
date                                                               
2025-10-20                       95.91  96.40  1166500.0  96.240   
2025-10-21                       96.13  96.80  1309100.0  96.600   
2025-10-22                       95.50  95.97  2308900.0  96.210   
2025-10-23                       95.98  96.81  4808100.0  96.490   
2025-10-24                       97.17  97.20  1337841.0  97.495   

           iShares Micro-Cap ETF                                            
                            open    high     low   close   volume     vwap  
date                                                                        
2025-10-20                156.06  157.99  155.02  157.76  19812.0  156.505  
2025-10-21                156.97  157.79  155.38  156.68  23200.0  156.585  
2025-10-22                155.43  155.95  150.98  153.44  27422.0  153.465  
2025-10-23                154.16  155.97  153.96  155.46  56206.0  154.965  
2025-10-24                156.60  159.24  156.60  158.45  16219.0  157.920  

[5 rows x 126 columns]

In [6]:
us_equity_assets = us_equity_data.columns.get_level_values(0).unique().tolist()
us_equity_derived = dd.TimeSeriesDerivedFields(price_data=us_equity_data.xs(us_equity_assets[0], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
us_equity_derived.columns = pd.MultiIndex.from_product([[us_equity_assets[0]], us_equity_derived.columns])
for i in range(1, len(us_equity_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=us_equity_data.xs(us_equity_assets[i], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
    temp.columns = pd.MultiIndex.from_product([[us_equity_assets[i]], temp.columns])
    us_equity_derived = pd.concat([us_equity_derived, temp], axis=1)

us_equity_derived.to_csv('data/ready/us_equity.csv')

Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 8203 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 16406/16406 [00:02<00:00, 6973.35it/s]


Generated 40 tsfresh features
Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 5621 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 11242/11242 [00:01<00:00, 6721.87it/s]


Generated 40 tsfresh features
Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 5514 out of 9002
Extracting tsfresh features...


...

Generated 40 tsfresh features
Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 6064 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 12128/12128 [00:01<00:00, 8109.91it/s]


Generated 40 tsfresh features
Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 5042 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 10084/10084 [00:01<00:00, 7916.60it/s]


Generated 40 tsfresh features


# US Treasury ETFs

In [8]:
us_equity_data = pd.read_csv('data/processed/us_equity_all_data.csv', index_col=0, header=[0, 1], parse_dates=True)
us_equity_data.index = pd.to_datetime(us_equity_data.index)
us_equity_data.rename(columns=us_equity_symbol_names, level=0, inplace=True)

us_treasuries = ['SPDR Bloomberg 1-3 Month T-Bill ETF', 'iShares 1-3 Year Treasury Bond ETF', 'iShares 7-10 Year Treasury Bond ETF']
col_mask = us_equity_data.columns.map(lambda x: x[0] in us_treasuries)

us_treasury_data = us_equity_data.loc[:, col_mask]
us_treasury_data.tail()

SPDR Bloomberg 1-3 Month T-Bill ETF                       \
                                          open   high    low  close   
date                                                                  
2025-10-20                               91.64  91.64  91.63  91.64   
2025-10-21                               91.65  91.65  91.64  91.64   
2025-10-22                               91.66  91.66  91.65  91.66   
2025-10-23                               91.66  91.67  91.66  91.67   
2025-10-24                               91.70  91.70  91.69  91.70   

                               iShares 1-3 Year Treasury Bond ETF         \
                volume    vwap                               open   high   
date                                                                       
2025-10-20   8215427.0  91.635                              83.10  83.10   
2025-10-21   6989615.0  91.645                              83.12  83.13   
2025-10-22  13191440.0  91.655                              83.10  83.14   
2025-10-23   7399500.0  91.665                              83.10  83.11   
2025-10-24   8743117.0  91.695                              83.12  83.13   

                                             \
              low  close     volume    vwap   
date                                          
2025-10-20  83.07  83.10  3331847.0  83.085   
2025-10-21  83.10  83.12  4726847.0  83.115   
2025-10-22  83.10  83.12  6952989.0  83.120   
2025-10-23  83.06  83.06  4405500.0  83.085   
2025-10-24  83.09  83.11  2293471.0  83.110   

           iShares 7-10 Year Treasury Bond ETF                       \
                                          open   high    low  close   
date                                                                  
2025-10-20                               97.45  97.53  97.37  97.52   
2025-10-21                               97.69  97.77  97.63  97.70   
2025-10-22                               97.65  97.77  97.56  97.72   
2025-10-23                               97.52  97.58  97.38  97.40   
2025-10-24                               97.52  97.54  97.33  97.49   

                                
                volume    vwap  
date                            
2025-10-20  10437930.0  97.450  
2025-10-21   7642000.0  97.700  
2025-10-22   6523872.0  97.665  
2025-10-23   6138339.0  97.480  
2025-10-24   7555376.0  97.435

In [9]:
us_treasury_assets = us_treasury_data.columns.get_level_values(0).unique().tolist()
us_treasury_derived = dd.TimeSeriesDerivedFields(price_data=us_treasury_data.xs(us_treasury_assets[0], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
us_treasury_derived.columns = pd.MultiIndex.from_product([[us_treasury_assets[0]], us_treasury_derived.columns])
for i in range(1, len(us_treasury_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=us_treasury_data.xs(us_treasury_assets[i], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
    temp.columns = pd.MultiIndex.from_product([[us_treasury_assets[i]], temp.columns])
    us_treasury_derived = pd.concat([us_treasury_derived, temp], axis=1)

us_treasury_derived.to_csv('data/ready/us_treasury.csv')

Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 4594 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 9188/9188 [00:01<00:00, 7040.31it/s]


Generated 40 tsfresh features
Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 5812 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 11624/11624 [00:01<00:00, 7989.97it/s]


Generated 40 tsfresh features
Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 5812 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 11624/11624 [00:01<00:00, 7013.53it/s]


Generated 40 tsfresh features


# International Equities

In [5]:
int_equity_data = pd.read_csv('data/processed/us_equity_all_data.csv', index_col=0, header=[0, 1], parse_dates=True)
int_equity_data.index = pd.to_datetime(int_equity_data.index)
int_equity_data.rename(columns=int_equity_symbol_names, level=0, inplace=True)

col_mask = int_equity_data.columns.map(lambda x: x[0] in int_equity_symbol_names.values())

int_equity_data = int_equity_data.loc[:, col_mask]
int_equity_data.tail()

Vanguard Total International Stock ETF                       \
                                             open   high    low  close   
date                                                                     
2025-10-20                                  74.57  74.98  74.56  74.93   
2025-10-21                                  74.48  74.52  74.21  74.22   
2025-10-22                                  74.34  74.54  73.93  74.28   
2025-10-23                                  74.48  74.81  74.46  74.71   
2025-10-24                                  74.97  75.06  74.86  74.95   

                               Vanguard FTSE Developed Markets ETF         \
                volume    vwap                                open   high   
date                                                                        
2025-10-20   4462625.0  74.770                               60.97  61.29   
2025-10-21   3818600.0  74.365                               60.90  60.93   
2025-10-22  12642700.0  74.235                               60.75  60.90   
2025-10-23   3015945.0  74.635                               60.81  61.13   
2025-10-24   3349497.0  74.960                               61.22  61.31   

                          ... iShares MSCI Japan ETF                    \
              low  close  ...                    low  close     volume   
date                      ...                                            
2025-10-20  60.97  61.24  ...                  83.29  83.54  5450900.0   
2025-10-21  60.66  60.69  ...                  82.43  82.57  5126791.0   
2025-10-22  60.44  60.71  ...                  81.84  82.18  7076378.0   
2025-10-23  60.81  61.02  ...                  81.98  82.22  3620433.0   
2025-10-24  61.12  61.22  ...                  82.42  82.50  3653247.0   

                   iShares MSCI India ETF                                   \
              vwap                   open   high    low  close      volume   
date                                                                         
2025-10-20  83.540                  54.53  54.77  54.53  54.74   8176700.0   
2025-10-21  82.645                  54.48  54.59  54.40  54.48   3190600.0   
2025-10-22  82.180                  55.37  55.50  55.13  55.29  10304600.0   
2025-10-23  82.160                  54.64  54.82  54.62  54.76   5730440.0   
2025-10-24  82.550                  54.59  54.61  54.49  54.51   5883402.0   

                    
              vwap  
date                
2025-10-20  54.650  
2025-10-21  54.495  
2025-10-22  55.315  
2025-10-23  54.720  
2025-10-24  54.550  

[5 rows x 48 columns]

In [6]:
int_equity_assets = int_equity_data.columns.get_level_values(0).unique().tolist()
int_equity_derived = dd.TimeSeriesDerivedFields(price_data=int_equity_data.xs(int_equity_assets[0], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
int_equity_derived.columns = pd.MultiIndex.from_product([[int_equity_assets[0]], int_equity_derived.columns])
for i in range(1, len(int_equity_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=int_equity_data.xs(int_equity_assets[i], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
    temp.columns = pd.MultiIndex.from_product([[int_equity_assets[i]], temp.columns])
    int_equity_derived = pd.concat([int_equity_derived, temp], axis=1)

Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 3669 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 7338/7338 [00:00<00:00, 7714.96it/s]


Generated 40 tsfresh features
Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 4554 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 9108/9108 [00:01<00:00, 6558.02it/s]


Generated 40 tsfresh features
Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 5152 out of 9002
Extracting tsfresh features...


...

Generated 40 tsfresh features
Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 7412 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 14824/14824 [00:01<00:00, 7538.75it/s]


Generated 40 tsfresh features
Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 3413 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 6826/6826 [00:00<00:00, 7422.87it/s]


Generated 40 tsfresh features


In [11]:
int_equity_derived.to_csv('data/ready/int_equity.csv')

# Commodities

In [7]:
comm_symbol_name_dict = fmp_comm.set_index('symbol')['name'].to_dict()
comm_symbol_name_dict.update({'Nickel': 'Nickel'})

commodity_data = pd.read_csv('data/processed/commodity_data.csv', index_col=0, header=[0,1])
commodity_data.index = pd.to_datetime(commodity_data.index)
commodity_data[('Nickel', 'vwap')] = np.nan

commodity_data.rename(columns=comm_symbol_name_dict, level=0, inplace=True)
commodity_data.tail()

Aluminum Futures                                              \
                       open     high      low    close  volume     vwap   
date                                                                      
2025-10-20          2689.25  2689.25  2689.25  2689.25    18.0  2689.25   
2025-10-21          2681.25  2681.25  2681.25  2681.25  3894.0  2681.25   
2025-10-22          2708.00  2708.00  2708.00  2708.00    44.0  2708.00   
2025-10-23          2768.75  2768.75  2768.75  2768.75   162.0  2768.75   
2025-10-24          2776.75  2776.75  2776.75  2776.75  2508.0  2776.75   

           Gold Futures                          ... Brent Crude Oil         \
                   open    high     low   close  ...             low  close   
date                                             ...                          
2025-10-20       4269.0  4398.0  4229.7  4359.4  ...           60.07  61.01   
2025-10-21       4371.0  4393.6  4093.0  4109.1  ...           60.34  61.32   
2025-10-22       4137.0  4175.0  4021.2  4065.4  ...           61.40  62.59   
2025-10-23       4114.8  4171.5  4079.6  4145.6  ...           63.85  65.99   
2025-10-24       4144.0  4159.0  4055.7  4137.8  ...           65.44  65.94   

                                Nickel                                       \
             volume     vwap      open      high       low     close volume   
date                                                                          
2025-10-20  50372.0  61.0150  15104.00  15217.13  15082.13  15217.13    NaN   
2025-10-21  45150.0  61.1650  15155.25  15242.50  15128.38  15164.13    NaN   
2025-10-22  46114.0  62.3825  15176.25  15196.13  15105.38  15152.38    NaN   
2025-10-23  87705.0  65.1075  15078.50  15372.88  15078.50  15337.13    NaN   
2025-10-24  87705.0  66.0050  15288.00  15366.88  15229.25  15336.38    NaN   

                 
           vwap  
date             
2025-10-20  NaN  
2025-10-21  NaN  
2025-10-22  NaN  
2025-10-23  NaN  
2025-10-24  NaN  

[5 rows x 72 columns]

In [8]:
commodity_assets = commodity_data.columns.get_level_values(0).unique().tolist()
commodity_equity_derived = dd.TimeSeriesDerivedFields(price_data=commodity_data.xs(commodity_assets[0], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
commodity_equity_derived.columns = pd.MultiIndex.from_product([[commodity_assets[0]], commodity_equity_derived.columns])
for i in range(1, len(commodity_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=commodity_data.xs(commodity_assets[i], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
    temp.columns = pd.MultiIndex.from_product([[commodity_assets[i]], temp.columns])
    commodity_equity_derived = pd.concat([commodity_equity_derived, temp], axis=1)

Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 594 out of 10926
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 1188/1188 [00:00<00:00, 7856.25it/s]

Generated 40 tsfresh features


Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 618 out of 10926
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 1236/1236 [00:00<00:00, 7838.60it/s]

...

Computing tsfresh features for columns: ['close', 'volume']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 590 out of 10926
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 1180/1180 [00:00<00:00, 5916.88it/s]


Generated 40 tsfresh features
Volume column exists but contains only NaN values. Skipping volume features.
Computing tsfresh features for columns: ['close']
Window size: 20, Shift periods: [1]
Valid windows for feature extraction: 103 out of 10926
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 103/103 [00:00<00:00, 4996.80it/s]

Generated 20 tsfresh features


In [12]:
commodity_equity_derived.to_csv('data/ready/commodity.csv')

# Macro Data

In [9]:
macro_data_daily = pd.read_csv('data/processed/macro_data_daily.csv', parse_dates=['date'], index_col='date')
macro_data_monthly = pd.read_csv('data/processed/macro_data_monthly.csv', parse_dates=['date'], index_col='date')
macro_data_quarterly = pd.read_csv('data/processed/macro_data_quarterly.csv', parse_dates=['date'], index_col='date')

macro_data_daily.columns = pd.MultiIndex.from_tuples([(f'{macro_codes_dict[col]} ({col})', f'{macro_units[col].split(',')[0]}, daily') for col in macro_data_daily.columns])
macro_data_monthly.columns = pd.MultiIndex.from_tuples([(f'{macro_codes_dict[col]} ({col})', f'{macro_units[col].split(',')[0]}, monthly') for col in macro_data_monthly.columns])
macro_data_quarterly.columns = pd.MultiIndex.from_tuples([(f'{macro_codes_dict[col]} ({col})', f'{macro_units[col].split(',')[0]}, quarterly') for col in macro_data_quarterly.columns])

# Forward fill less frequently updated macro data
macro_data_monthly_ffill = macro_data_monthly.reindex(macro_data_daily.index).ffill()
macro_data_quarterly_ffill = macro_data_quarterly.reindex(macro_data_daily.index).ffill()

macro_data_all = pd.concat([macro_data_daily, macro_data_monthly_ffill, macro_data_quarterly_ffill], axis=1)
macro_data_all.tail()

,Secured Overnight Financing Rate (SOFR),1-Month Treasury Rate (DGS1MO),3-Month Treasury Rate (DGS3MO),6-Month Treasury Rate (DGS6MO),1-Year Treasury Rate (DGS1),2-Year Treasury Rate (DGS2),3-Year Treasury Rate (DGS3),5-Year Treasury Rate (DGS5),7-Year Treasury Rate (DGS7),10-Year Treasury Rate (DGS10),...,Real Gross Domestic Product (GDPC1),Real GDP Per Capita (A939RX0Q048SBEA),Federal Government Debt to GDP Ratio (GFDEGDQ188S),Gross Private Domestic Investment (GPDI),Real Gross Private Domestic Investment (GPDIC1),Foreign Direct Investment in the United States (BOGZ1FU263092001Q),Balance Sheet - Total Assets (QBPBSTAS),Federal Debt Held by Federal Reserve Banks (FDHBFRBN),Federal Debt Held by the Public (FYGFDPUN),Federal Debt Held by Foreign Investors (FDHBFIN)
,"Percent, daily","Percent, daily","Percent, daily","Percent, daily","Percent, daily","Percent, daily","Percent, daily","Percent, daily","Percent, daily","Percent, daily",...,"Billions of Chained 2017 Dollars, quarterly","Chained 2017 Dollars, quarterly","Percent of GDP, quarterly","Billions of Dollars, quarterly","Billions of Chained 2017 Dollars, quarterly","Millions of Dollars, quarterly","Millions of Dollars, quarterly","Billions of Dollars, quarterly","Millions of Dollars, quarterly","Billions of Dollars, quarterly"
date,,,,,,,,,,,,,,,,,,,,,
2025-10-20,4.16,4.15,3.97,3.78,3.55,3.46,3.47,3.58,3.77,4.00,...,23770.976,69499.0,118.78171,5358.632,4382.819,76933.0,2.498867e+07,4532.664,28976177.0,9127.7
2025-10-21,4.23,4.12,3.96,3.78,3.56,3.45,3.46,3.56,3.74,3.98,...,23770.976,69499.0,118.78171,5358.632,4382.819,76933.0,2.498867e+07,4532.664,28976177.0,9127.7
2025-10-22,4.21,4.11,3.96,3.78,3.55,3.45,3.44,3.56,3.74,3.97,...,23770.976,69499.0,118.78171,5358.632,4382.819,76933.0,2.498867e+07,4532.664,28976177.0,9127.7
2025-10-23,4.24,4.12,3.95,3.78,3.59,3.48,3.49,3.61,3.79,4.01,...,23770.976,69499.0,118.78171,5358.632,4382.819,76933.0,2.498867e+07,4532.664,28976177.0,9127.7
2025-10-24,4.24,4.11,3.93,3.76,3.58,3.48,3.49,3.61,3.79,4.02,...,23770.976,69499.0,118.78171,5358.632,4382.819,76933.0,2.498867e+07,4532.664,28976177.0,9127.7


In [ ]:
macro_data_all.to_csv('data/ready/macro_data_all.csv')

# 